# 01b: Data Quality Assessment

**Purpose:** Deep dive into data quality, outliers, and influential observations

**Dataset:** COMPAS (cleaned)

**Date:** 2025-11-08

---

## Overview

### Purpose
- Identify outliers in continuous features
- Detect influential observations
- Assess multicollinearity (VIF)
- Check distribution normality
- Generate quality assurance report

### Inputs
- Cleaned COMPAS data from `02a_data_cleaning.ipynb`

### Outputs
- Outlier analysis → `results/figures/exploratory/outliers_analysis.png`
- VIF report → `results/tables/vif_multicollinearity.csv`
- Quality report → `data/metadata/quality_assessment.json`

### Runtime: 2-3 minutes

---

In [ ]:
# Setup
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / "src"))

# Directories
PROCESSED_DIR = project_root / "data" / "processed"
FIGURES_DIR = project_root / "results" / "figures" / "exploratory"
TABLES_DIR = project_root / "results" / "tables"
METADATA_DIR = project_root / "data" / "metadata"

for d in [FIGURES_DIR, TABLES_DIR, METADATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-darkgrid')
print("✓ Setup complete")

## 1. Load Cleaned Data

In [ ]:
# Load cleaned data
df = pd.read_parquet(PROCESSED_DIR / "compas_cleaned.parquet")
print(f"Loaded {len(df):,} samples")

# Identify continuous features
continuous_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'two_year_recid' in continuous_cols:
    continuous_cols.remove('two_year_recid')

print(f"Continuous features: {len(continuous_cols)}")

## 2. Outlier Detection

### 2.1 IQR Method

In [ ]:
# Detect outliers using IQR method
outlier_summary = []

for col in continuous_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    n_outliers = len(outliers)
    pct_outliers = n_outliers / len(df) * 100
    
    outlier_summary.append({
        'feature': col,
        'n_outliers': n_outliers,
        'pct_outliers': pct_outliers,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('n_outliers', ascending=False)
print("Outlier detection (IQR method):")
display(outlier_df)

### 2.2 Visualize Outliers

In [ ]:
# Box plots for outlier visualization
n_cols = 3
n_rows = (len(continuous_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten() if len(continuous_cols) > 1 else [axes]

for i, col in enumerate(continuous_cols):
    axes[i].boxplot(df[col].dropna(), vert=True)
    axes[i].set_title(f'{col}', fontweight='bold')
    axes[i].set_ylabel('Value')
    axes[i].grid(alpha=0.3)

for i in range(len(continuous_cols), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'outliers_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: outliers_boxplots.png")

## 3. Multicollinearity Assessment

Calculate Variance Inflation Factor (VIF) to detect multicollinearity.

In [ ]:
# Calculate VIF
vif_data = []

# Prepare data (drop NaNs, standardize)
X = df[continuous_cols].dropna()

for i, col in enumerate(continuous_cols):
    vif = variance_inflation_factor(X.values, i)
    vif_data.append({'feature': col, 'VIF': vif})

vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)

print("Variance Inflation Factors:")
display(vif_df)

# Interpretation
high_vif = vif_df[vif_df['VIF'] > 10]
if len(high_vif) > 0:
    print(f"\n⚠ High multicollinearity detected ({len(high_vif)} features with VIF > 10):")
    for _, row in high_vif.iterrows():
        print(f"  - {row['feature']}: VIF = {row['VIF']:.2f}")
else:
    print("\n✓ No severe multicollinearity (all VIF < 10)")

# Save
vif_df.to_csv(TABLES_DIR / 'vif_multicollinearity.csv', index=False)
print("\n✓ Saved: vif_multicollinearity.csv")

## 4. Normality Tests

In [ ]:
# Shapiro-Wilk test for normality (on sample if n > 5000)
normality_results = []

for col in continuous_cols:
    data = df[col].dropna()
    
    # Sample if too large
    if len(data) > 5000:
        data = data.sample(5000, random_state=42)
    
    stat, p_value = stats.shapiro(data)
    is_normal = p_value > 0.05
    
    normality_results.append({
        'feature': col,
        'shapiro_stat': stat,
        'p_value': p_value,
        'is_normal': is_normal
    })

normality_df = pd.DataFrame(normality_results)
print("Normality tests (Shapiro-Wilk):")
display(normality_df)

n_normal = normality_df['is_normal'].sum()
print(f"\n{n_normal} of {len(continuous_cols)} features appear normally distributed")

## 5. Quality Assessment Report

In [ ]:
# Compile quality assessment
quality_report = {
    'notebook': '01b_data_quality_assessment.ipynb',
    'n_samples': len(df),
    'n_features': len(continuous_cols),
    'outliers': {
        'method': 'IQR (1.5 * IQR)',
        'features_with_outliers': int((outlier_df['n_outliers'] > 0).sum()),
        'max_outlier_pct': float(outlier_df['pct_outliers'].max()),
        'summary': outlier_df.to_dict('records')
    },
    'multicollinearity': {
        'max_vif': float(vif_df['VIF'].max()),
        'features_high_vif': vif_df[vif_df['VIF'] > 10]['feature'].tolist(),
        'vif_summary': vif_df.to_dict('records')
    },
    'normality': {
        'n_normal': int(n_normal),
        'n_non_normal': int(len(continuous_cols) - n_normal),
        'summary': normality_df.to_dict('records')
    },
    'recommendations': {
        'transformations': 'Consider log transformation for non-normal count features',
        'outliers': 'Most outliers appear legitimate (high prior counts), do not remove',
        'multicollinearity': 'No severe issues detected' if len(high_vif) == 0 else 'Consider removing high VIF features'
    }
}

# Save
with open(METADATA_DIR / 'quality_assessment.json', 'w') as f:
    json.dump(quality_report, f, indent=2)

print("✓ Saved: quality_assessment.json")
print("\n" + "="*60)
print("DATA QUALITY ASSESSMENT COMPLETE")
print("="*60)
print(f"✓ Outliers detected but appear legitimate (criminal history)")
print(f"✓ No severe multicollinearity (max VIF: {vif_df['VIF'].max():.2f})")
print(f"✓ {n_normal}/{len(continuous_cols)} features normally distributed")
print("\nNext: 01d_descriptive_statistics.ipynb")